# Burke & Emerick (2016)


Illinois county-level LOCA2 indicators. Keep this notebook and `loca2_common.py` in the same folder. Open Jupyter in that folder and run the cells from top to bottom. Each notebook uses its own kernel.

Existing files are skipped (`OVERWRITE = False`). County aggregation uses cosine-latitude area correction.

This workflow requires both relative-humidity inputs as well as temperature and precipitation.

Go to loca2_common.py to set your input and output path. 

Indicators calculated for Growing Season:Annual Growing Degree Days, Annual Extreme Degree Days and Annual Total Precipitation 

Growing season: March 1 through August 31.

## 1. Import shared functions

Required packages: numpy, xarray, dask, netCDF4, cftime, pandas, geopandas, regionmask, and tqdm. Use your existing Keeling environment.

In [1]:
from pathlib import Path
import numpy as np
import xarray as xr
import loca2_common as common
from loca2_common import (
    aggregate_to_illinois_counties,
    annual_sum,
    bounded_degree_days,
    degree_days_above_threshold,
    select_growing_season,
)

## 2. Paper settings

In [2]:
BURKE_SEASON_START = '03-01'
BURKE_SEASON_END = '08-31'
PAPER_NAME = 'Burke & Emerick (2016)'
REQUIRED_VARIABLES = ('tasmin', 'tasmax', 'pr')
OVERWRITE = False

## 3. Check paths and available runs

This cell lists the workload and loads county boundaries. It does not write climate output.

In [3]:
output_directory = common.PAPER_OUTPUT_ROOT / common.paper_directory_name(PAPER_NAME)
jobs = common.discover_paper_runs(REQUIRED_VARIABLES)
existing = sum(
    (output_directory / (
        f"{common.safe_token(PAPER_NAME)}_"
        f"{common.safe_token(model)}_{common.safe_token(member)}_{scenario}.nc"
    )).exists()
    for model, member, scenario in jobs
)
print("Input root:", common.PAPER_INPUT_ROOT)
print("Output folder:", output_directory)
print("Eligible runs:", len(jobs))
print("Existing files to skip:", existing if not OVERWRITE else 0)
print("Runs to process:", len(jobs) - existing if not OVERWRITE else len(jobs))
if not jobs:
    raise FileNotFoundError("No eligible runs found. Check input paths and required variables.")
counties = common.load_counties()
print("Counties:", len(counties))

Input root: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois
Output folder: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois/temp/Burke & Emerick (2016)
Eligible runs: 156
Existing files to skip: 0
Runs to process: 156
Counties: 102


## 4. Paper indicator calculation

In [ ]:
def calculate_burke_emerick(inputs):
    """
    Calculate Burke and Emerick indicators and aggregate them
    to Illinois counties.

    Output dimensions:
        year × county
    """
    required_variables = {'tasmin', 'tasmax', 'pr'}
    missing_variables = required_variables - set(inputs)
    if missing_variables:
        raise KeyError(f'Missing required Burke and Emerick inputs: {sorted(missing_variables)}')
    original_time_sizes = {'tasmin': inputs['tasmin'].sizes['time'], 'tasmax': inputs['tasmax'].sizes['time'], 'pr': inputs['pr'].sizes['time']}
    tasmin_c, tasmax_c, precipitation_in = xr.align(inputs['tasmin'], inputs['tasmax'], inputs['pr'], join='inner', copy=False)
    aligned_time_size = tasmin_c.sizes['time']
    if aligned_time_size == 0:
        raise ValueError('No shared dates were found among tasmin, tasmax, and precipitation.')
    removed_dates = {variable_name: original_size - aligned_time_size for variable_name, original_size in original_time_sizes.items()}
    if any((count > 0 for count in removed_dates.values())):
        print(f'Time alignment retained shared dates only. Removed dates: {removed_dates}')
    if not (tasmin_c.indexes['time'].equals(tasmax_c.indexes['time']) and tasmin_c.indexes['time'].equals(precipitation_in.indexes['time'])):
        raise ValueError('Time coordinates remain inconsistent after inner alignment.')
    precipitation_cm = (precipitation_in * 2.54).rename('daily_precipitation_cm')
    precipitation_cm.attrs.update(units='cm day-1', long_name='daily precipitation', conversion='converted from inches using 1 inch = 2.54 cm')
    tmin_c = select_growing_season(tasmin_c, BURKE_SEASON_START, BURKE_SEASON_END)
    tmax_c = select_growing_season(tasmax_c, BURKE_SEASON_START, BURKE_SEASON_END)
    precip_cm = select_growing_season(precipitation_cm, BURKE_SEASON_START, BURKE_SEASON_END)
    if tmin_c.sizes['time'] == 0:
        raise ValueError('No March-August dates remained after time alignment.')
    tmin_c, tmax_c, precip_cm = xr.align(tmin_c, tmax_c, precip_cm, join='exact', copy=False)
    dd_0_29 = bounded_degree_days(tmin_c=tmin_c, tmax_c=tmax_c, lower_c=0.0, upper_c=29.0)
    dd_above_29 = degree_days_above_threshold(tmin_c, tmax_c, threshold_c=29.0)
    dd_0_28 = bounded_degree_days(tmin_c=tmin_c, tmax_c=tmax_c, lower_c=0.0, upper_c=28.0)
    dd_above_28 = degree_days_above_threshold(tmin_c, tmax_c, threshold_c=28.0)
    result = xr.Dataset({'degree_days_0_29C': annual_sum(dd_0_29), 'degree_days_above_29C': annual_sum(dd_above_29), 'degree_days_0_28C': annual_sum(dd_0_28), 'degree_days_above_28C': annual_sum(dd_above_28), 'growing_season_precipitation_cm': annual_sum(precip_cm)})
    degree_day_metadata = {'degree_days_0_29C': 'March-August sinusoidal degree days between 0 and 29 degrees Celsius', 'degree_days_above_29C': 'March-August sinusoidal degree days above 29 degrees Celsius', 'degree_days_0_28C': 'March-August sinusoidal degree days between 0 and 28 degrees Celsius', 'degree_days_above_28C': 'March-August sinusoidal degree days above 28 degrees Celsius'}
    for variable_name, long_name in degree_day_metadata.items():
        result[variable_name].attrs.update(units='degree_Celsius day', long_name=long_name, temporal_aggregation='sum from March 1 through August 31', temperature_method='sinusoidal daily temperature interpolation')
    result['growing_season_precipitation_cm'].attrs.update(units='cm', long_name='March-August total precipitation', temporal_aggregation='sum of daily precipitation from March 1 through August 31')
    result.attrs.update(paper='Burke & Emerick (2016)', temporal_resolution='annual', growing_season_start=BURKE_SEASON_START, growing_season_end=BURKE_SEASON_END, temperature_method='sinusoidal daily temperature interpolation', precipitation_input_conversion='daily precipitation converted from inches to centimeters using 1 inch = 2.54 cm', time_alignment='shared dates among tasmin, tasmax, and precipitation')
    result = aggregate_to_illinois_counties(dataset=result, counties=counties)
    county_precipitation_cm = result['growing_season_precipitation_cm']
    result['precipitation_up_to_42cm'] = county_precipitation_cm.clip(min=0.0, max=42.0)
    result['precipitation_exceeding_42cm'] = (county_precipitation_cm - 42.0).clip(min=0.0)
    result['precipitation_up_to_50cm'] = county_precipitation_cm.clip(min=0.0, max=50.0)
    result['precipitation_exceeding_50cm'] = (county_precipitation_cm - 50.0).clip(min=0.0)
    precipitation_metadata = {'precipitation_up_to_42cm': {'long_name': 'county growing-season precipitation up to 42 centimeters', 'calculation': 'minimum(P, 42)', 'threshold_cm': 42.0}, 'precipitation_exceeding_42cm': {'long_name': 'county growing-season precipitation exceeding 42 centimeters', 'calculation': 'maximum(P - 42, 0)', 'threshold_cm': 42.0}, 'precipitation_up_to_50cm': {'long_name': 'county growing-season precipitation up to 50 centimeters', 'calculation': 'minimum(P, 50)', 'threshold_cm': 50.0}, 'precipitation_exceeding_50cm': {'long_name': 'county growing-season precipitation exceeding 50 centimeters', 'calculation': 'maximum(P - 50, 0)', 'threshold_cm': 50.0}}
    for variable_name, metadata in precipitation_metadata.items():
        result[variable_name].attrs.update(units='cm', long_name=metadata['long_name'], calculation=metadata['calculation'], threshold_cm=metadata['threshold_cm'], threshold_application='applied to county-average March-August precipitation after spatial aggregation')
    result.attrs.update(spatial_resolution='Illinois county', precipitation_thresholds='42 cm and 50 cm', precipitation_threshold_order='county-average growing-season precipitation calculated first; threshold transformations applied second', weighting='cosine-latitude grid-cell area correction only; no LandScan, population, or crop weighting')
    return result

## 5. Generate output

Existing matching filenames are skipped. A skipped file is not recalculated or checked for completeness.

In [ ]:
common.run_paper(
    paper_name=PAPER_NAME,
    required_variables=REQUIRED_VARIABLES,
    calculation_function=calculate_burke_emerick,
    overwrite=OVERWRITE,
)

## 6. Inspect an output file

In [ ]:
files = sorted(output_directory.glob(f"{common.safe_token(PAPER_NAME)}_*.nc"))
if files:
    with xr.open_dataset(files[0], decode_timedelta=False) as dataset:
        print("Example:", files[0].name)
        print("Dimensions:", dict(dataset.sizes))
        print("Variables:", list(dataset.data_vars))
        print("Spatial resolution:", dataset.attrs.get("spatial_resolution"))
        assert dataset.sizes.get("county") == 102, "Output is not a 102-county dataset."
else:
    print("No output files found. Review the processing messages above.")